# One earnings gap. Six different stories.

**Chapter 2 · Guided simulation lab · 30–45 minutes · Complete beginners welcome**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cmg777/intro2causal/blob/main/notebooks_colab/explore-chapter-2-simulations.ipynb)

> **Can a school appear to raise earnings even when its true effect is zero?**

Imagine comparing earnings of people who attended private and other schools. Attendance may reflect ability, family resources, and motivation, all of which could also affect earnings. In real observational data, we do not know the true school effect. In this lab, we construct six worlds and choose that effect ourselves. Each world contains 1,000 simulated students, and each changes a specific feature of selection or confounding. These are the six scenarios in the Chapter 2 Python exercises, reproduced with the same equations and seed resets. None is a real student dataset, and none uses randomized school attendance. That distinction matters: changing the treatment effect is different from making treatment random.

**You will learn to:** distinguish a raw gap from a known effect; visualize selection on ability; explain why reversing selection can reverse bias.

**Predict before running:** Could privately educated students earn more even if private education had no causal effect? Write one explanation.

**Route:** construct six worlds → inspect one → compare gaps → change one view → explain the causal lesson.

## Your first few minutes in Colab · 3 minutes

You do not need prior Python experience. A **cell** is one block of text or code. Read the text, then press the **▶** beside the next code cell (or Shift+Enter). Its output appears underneath. Run cells in order: later cells use tables created earlier.

Choose **File → Save a copy in Drive** if you want to keep your changes. If a name is “not defined,” run the earlier setup cells again. **Runtime → Restart session and run all** gives you a fresh start. An error does not mean you have damaged the data: this notebook only reads it.

Keep a short learning log in a text cell. Before each chart, make a prediction. Afterward, write one observation with a number and one question that the chart cannot answer.

**Chart controls:** hover for values, click legend labels to hide/show a group, drag to zoom, and double-click to reset. A chart’s menu also offers a reset/autoscale control.

The setup installs Plotly Express if needed. `import` makes a library available; `pd` and `px` are short names for pandas (tables) and Plotly Express (charts).

In [ ]:
%pip install -q "plotly>=6,<7"

In [ ]:
import pandas as pd
import plotly.express as px
from IPython.display import display

# One consistent chart style; no plotting experience needed.
px.defaults.template = "plotly_white"
px.defaults.color_continuous_scale = "Blues"
px.defaults.color_discrete_sequence = ["#2563eb", "#7c3aed", "#059669", "#64748b"]
import numpy as np
scenarios = {}
true_effects = {}

## Build six small worlds · 8 minutes

`np.random.seed(42)` restarts a reproducible sequence of random draws. Each scenario restarts it, as the chapter does. Ability is normally distributed around 50. The assignment equation converts characteristics to a probability between 0 and 1; `binomial(1, probability)` draws private attendance as 0 or 1. Earnings combine a school effect, other influences, and random noise.

Notice **which variables enter both assignment and earnings**. That is the central reasoning task. The six short cells expose the construction; running them is enough to continue.

### Scenario 1 · Baseline selection

Higher ability raises the chance of private attendance; the true school effect is $5,000.

Predict how the observed earnings gap will change. Run this short construction cell; you do not need to memorize the equations.

In [ ]:
np.random.seed(42)
n = 1000
ability = np.random.normal(50, 10, n)
prob_private = 1 / (1 + np.exp(-((ability - 50) / 5)))
private = np.random.binomial(1, prob_private)
earnings = 30000 + 5000 * private + 800 * ability + np.random.normal(0, 5000, n)
scenarios["1. Baseline selection"] = pd.DataFrame({"ability": ability, "private": private, "earnings": earnings})
true_effects["1. Baseline selection"] = 5000

### Scenario 2 · No treatment effect

Keep ability-based selection but remove the school effect. A raw gap can remain even when the true effect is zero.

Predict how the observed earnings gap will change. Run this short construction cell; you do not need to memorize the equations.

In [ ]:
np.random.seed(42)
n = 1000
ability = np.random.normal(50, 10, n)
prob_private = 1 / (1 + np.exp(-((ability - 50) / 5)))
private = np.random.binomial(1, prob_private)
earnings = 30000 + 0 * private + 800 * ability + np.random.normal(0, 5000, n)
scenarios["2. No treatment effect"] = pd.DataFrame({"ability": ability, "private": private, "earnings": earnings})
true_effects["2. No treatment effect"] = 0

### Scenario 3 · Stronger selection

Divide ability by 2 instead of 5 in the assignment equation, making selection steeper. The effect remains $5,000.

Predict how the observed earnings gap will change. Run this short construction cell; you do not need to memorize the equations.

In [ ]:
np.random.seed(42)
n = 1000
ability = np.random.normal(50, 10, n)
prob_private = 1 / (1 + np.exp(-((ability - 50) / 2)))
private = np.random.binomial(1, prob_private)
earnings = 30000 + 5000 * private + 800 * ability + np.random.normal(0, 5000, n)
scenarios["3. Stronger selection"] = pd.DataFrame({"ability": ability, "private": private, "earnings": earnings})
true_effects["3. Stronger selection"] = 5000

### Scenario 4 · Add family income

Income now influences both attendance and earnings. The effect remains $5,000.

Predict how the observed earnings gap will change. Run this short construction cell; you do not need to memorize the equations.

In [ ]:
np.random.seed(42)
n = 1000
ability = np.random.normal(50, 10, n)
income = np.random.normal(60000, 20000, n)
prob_private = 1 / (1 + np.exp(-((ability - 50) / 5 + (income - 60000) / 20000)))
private = np.random.binomial(1, prob_private)
earnings = 10000 + 5000 * private + 800 * ability + 0.3 * income + np.random.normal(0, 5000, n)
scenarios["4. Add family income"] = pd.DataFrame({"ability": ability, "private": private, "earnings": earnings, "income": income})
true_effects["4. Add family income"] = 5000

### Scenario 5 · Reverse selection

Higher ability now makes private attendance less likely. The effect remains $5,000, but the direction of confounding reverses.

Predict how the observed earnings gap will change. Run this short construction cell; you do not need to memorize the equations.

In [ ]:
np.random.seed(42)
n = 1000
ability = np.random.normal(50, 10, n)
prob_private = 1 / (1 + np.exp(((ability - 50) / 5)))
private = np.random.binomial(1, prob_private)
earnings = 30000 + 5000 * private + 800 * ability + np.random.normal(0, 5000, n)
scenarios["5. Reverse selection"] = pd.DataFrame({"ability": ability, "private": private, "earnings": earnings})
true_effects["5. Reverse selection"] = 5000

### Scenario 6 · Three confounders

Ability, income, and motivation each influence attendance and earnings. The effect remains $5,000.

Predict how the observed earnings gap will change. Run this short construction cell; you do not need to memorize the equations.

In [ ]:
np.random.seed(42)
n = 1000
ability = np.random.normal(50, 10, n)
income = np.random.normal(60000, 20000, n)
motivation = np.random.normal(5, 2, n)
prob_private = 1 / (1 + np.exp(-((ability - 50) / 5 + (income - 60000) / 20000 + (motivation - 5) / 2)))
private = np.random.binomial(1, prob_private)
earnings = 10000 + 5000 * private + 800 * ability + 0.3 * income + 2000 * motivation + np.random.normal(0, 5000, n)
scenarios["6. Three confounders"] = pd.DataFrame({"ability": ability, "private": private, "earnings": earnings, "income": income, "motivation": motivation})
true_effects["6. Three confounders"] = 5000

## Meet one world · 4 minutes

Each row is a simulated student. `private` is an attendance indicator (1 private, 0 other); `ability` is a constructed score; `earnings` is annual earnings in simulated dollars. Here earnings are **not logs**, unlike some Chapter 6 CSVs.

`head` shows rows; the next table checks types and missingness. `describe` summarizes numeric variables. The attendance counts describe a category, even though its storage type is numeric.

In [ ]:
baseline = scenarios["1. Baseline selection"]
display(baseline.head())
display(pd.DataFrame({"type": baseline.dtypes.astype(str), "missing": baseline.isna().sum()}))
display(baseline.describe())
display(baseline["private"].value_counts().rename("Students"))

## Visual investigation 1 · who attends which school?

**Predict:** will ability be similarly distributed in the two attendance groups?

`map` turns 0 and 1 into readable labels. A box shows the middle half of the ability distribution, with its median inside.

In [ ]:
baseline = baseline.assign(school=baseline["private"].map({0: "Other school", 1: "Private school"}))
fig = px.box(baseline, x="school", y="ability", color="school", points=False, labels={"school": "Attendance", "ability": "Simulated ability"}, title="Selection is visible before comparing earnings")
fig.show()

**Explore:** hover over the boxes. Explain why their difference matters if ability also affects earnings.

**Checkpoint:** higher ability makes private attendance more likely in this world. A raw earnings comparison therefore bundles the school effect with an ability difference.

## Visual investigation 2 · compare what we see with what we know

For each world, subtract mean earnings among other-school students from mean earnings among private-school students. The `for` loop repeats the same calculation for each named table. A list collects the results; `DataFrame` makes that list into a table.

**Predict:** in which world should the observed gap be most misleading?

In [ ]:
rows = []
for name, students in scenarios.items():
    means = students.groupby("private")["earnings"].mean()
    rows.append({"scenario": name, "Observed gap": means.loc[1] - means.loc[0], "Known school effect": true_effects[name]})
comparison = pd.DataFrame(rows)
long = comparison.melt(id_vars="scenario", var_name="quantity", value_name="dollars")
fig = px.scatter(long, x="dollars", y="scenario", color="quantity", symbol="quantity", labels={"dollars": "Annual earnings difference (simulated dollars)", "scenario": "World", "quantity": "Quantity"}, title="A raw gap is not the school effect")
fig.add_vline(x=0, line_dash="dot")
fig.show()

**Explore:** hover over both points for the zero-effect world and the reversed-selection world. State the observed gap and known effect separately.

**Checkpoint:** the zero-effect world can display a positive earnings gap because of selection. Reversing selection can push the observed gap below the positive school effect. Random noise also contributes to finite-sample differences.

## Visual investigation 3 · connect the discrepancy to selection

**Predict:** which world puts lower-ability students into private education?

We now repeat the difference-in-means calculation for ability, which exists in every world. This is evidence about one confounder, not a complete explanation of the multi-confounder worlds.

In [ ]:
ability_rows = []
for name, students in scenarios.items():
    means = students.groupby("private")["ability"].mean()
    ability_rows.append({"scenario": name, "ability_gap": means.loc[1] - means.loc[0]})
ability_gaps = pd.DataFrame(ability_rows)
fig = px.bar(ability_gaps, x="ability_gap", y="scenario", orientation="h", labels={"ability_gap": "Private minus other: mean ability", "scenario": "World"}, title="The direction of selection can change")
fig.add_vline(x=0, line_dash="dot")
fig.show()

**Explore:** compare the sign of the ability gap with the observed-minus-known earnings discrepancy. Why is ability alone insufficient in worlds 4 and 6?

**Checkpoint:** the direction of selection changes the interpretation of the raw comparison. Knowing the true effect is a privilege of a simulation, not a feature of real observational data.

## Make it yours · 8 minutes

### 1. Modify an example

Change the ability box plot to use scenario 5. Predict how the group ordering will change.

Copy that chart cell before editing, so you can compare the original and modified outputs. Say what changed in your question as well as in your code.

### 2. Try a small investigation

Plot earnings against ability in the zero-effect world, colored by school attendance.

Write a prediction first. The optional starter cell below is safe to leave unchanged; later cells do not depend on your answer.

In [ ]:
# Your investigation goes here.
# Describe your expected pattern before writing code.

<details>
<summary>Hint 1 · choose the right evidence</summary>

Choose scenarios["2. No treatment effect"].

</details>

<details>
<summary>Hint 2 · a Python starting point</summary>

Map private to readable school labels, then use px.scatter with x="ability" and y="earnings".

</details>

<details>
<summary>Optional worked solution · open after trying</summary>

<pre><code>zero = scenarios[&quot;2. No treatment effect&quot;].copy()
zero[&quot;school&quot;] = zero[&quot;private&quot;].map({0: &quot;Other school&quot;, 1: &quot;Private school&quot;})
fig = px.scatter(zero, x=&quot;ability&quot;, y=&quot;earnings&quot;, color=&quot;school&quot;, opacity=0.5, labels={&quot;ability&quot;: &quot;Simulated ability&quot;, &quot;earnings&quot;: &quot;Annual earnings (simulated dollars)&quot;, &quot;school&quot;: &quot;Attendance&quot;})
fig.show()</code></pre>

School attendance sorts students along ability even when its direct earnings effect is zero. The color pattern does not imply that the school caused their earnings.

Copy the code into the starter cell to run it.

</details>

## Return to your prediction · 4 minutes

Write an evidence note: one observed gap, its known effect, one source of their difference, and one assumption you would need in real data. Explain why a zero effect does not imply a zero observed gap.

**Next question:** When can regression adjustment separate a treatment effect from selection—and what happens when a confounder remains unobserved?

[Continue with Chapter 2: Regression](https://cmg777.github.io/intro2causal/book/_book/notebooks_quarto/02-regression.html) · [Return to the case collection](https://cmg777.github.io/intro2causal/data-dictionary.html#chapter-2-simulations)

**Construction:** exact six Python simulation variants in the chapter, seed 42 restarted for each, 1,000 students per world, normal earnings noise with SD $5,000. The known effect is $0 in scenario 2 and $5,000 in scenarios 1 and 3–6. No survey weights apply. The scenarios change equations, not just random samples.